## Extracting Past data using yahoo finance

In [1]:
import yfinance as yf
import pandas as pd
# Pull HDFC Bank data (use .NS for NSE, .BO for BSE)
ticker = yf.Ticker('HDFCBANK.NS')
# The .T transposes so dates are rows, metrics are columns
income_stmt   = ticker.financials.T
balance_sheet = ticker.balance_sheet.T
cash_flow     = ticker.cashflow.T
# Save raw files — never modify these
income_stmt.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\income_statement.csv')
balance_sheet.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\balance_sheet.csv')
cash_flow.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\cash_flow.csv')
# Print available columns to understand what data you have
print('Income Statement columns:')
print(income_stmt.columns.tolist())
print(f'Years available: {income_stmt.index.tolist()}')

Income Statement columns:
['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Minority Interests', 'Net Income Including Noncontrolling Interests', 'Net Income Extraordinary', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Non Operating Income Expenses', 'Special Income Charges', 'Gain On Sale Of Security', 'Depreciation Amortization Depletion Income Statement', 'Depreciation And Amortization In Income Statement', 'Amortization', 'Amortization Of Intangibles Income Statement', 'Depreciation Income State

### Since yfinance gives only 4 years of data and its not enough to train ML model for forecasting I switched to Screener for Data extraction 
Another benefit of doing so is Screener gives Data in RBI's recommeded format which is relevant for HDFC as it's listed in India 

In [2]:
import pandas as pd
import os

# Create the directory
#os.makedirs('data/raw', exist_ok=True)

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Balance Sheet
balance_sheet = None
for i, table in enumerate(tables):
    if "Share Capital" in table.iloc[:, 0].values:
        balance_sheet = table
        break

# Export the 10-year historical data to CSV
if balance_sheet is not None:
    # Set the first column (line items) as the index before saving
    balance_sheet.set_index(balance_sheet.columns[0], inplace=True)
    balance_sheet.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_balance_sheet.csv')
    print("Success! 10-year Balance Sheet saved to data/raw/screener_balance_sheet.csv.")
else:
    print("Error: Could not locate the Balance Sheet table.")

Error: Could not locate the Balance Sheet table.


In [3]:
import pandas as pd
import os

# Create the directory
#os.makedirs('data/raw', exist_ok=True)

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Balance Sheet using Bank-specific line items
balance_sheet = None
for i, table in enumerate(tables):
    # Convert the first column to strings to check for keywords safely
    first_col = table.iloc[:, 0].astype(str)
    
    # Banks list "Deposits" and "Capital" instead of generic "Share Capital"
    if first_col.str.contains('Deposits', case=False, na=False).any() or \
       first_col.str.contains('Capital', case=False, na=False).any():
        balance_sheet = table
        print(f"Balance sheet found at table index {i}!")
        break

# Export the 10-year historical data to CSV (TRANSPOSED)
if balance_sheet is not None:
    # 1. Set the first column (line items) as the index
    balance_sheet.set_index(balance_sheet.columns[0], inplace=True)
    
    # 2. Transpose the dataframe so Dates become rows and Line Items become columns
    balance_sheet_transposed = balance_sheet.T
    
    # 3. Export to CSV
    balance_sheet_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_balance_sheet.csv')
    print("Success! Transposed 10-year Balance Sheet saved to data/raw/screener_balance_sheet.csv.")
else:
    print("Error: Could not locate the Balance Sheet table.")

Balance sheet found at table index 6!
Success! Transposed 10-year Balance Sheet saved to data/raw/screener_balance_sheet.csv.


In [4]:
import pandas as pd
import os

# Create the directory just in case
#os.makedirs('data/raw', exist_ok=True)

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Cash Flow Statement
cash_flow = None
for i, table in enumerate(tables):
    first_col = table.iloc[:, 0].astype(str)
    
    # Looking for standard Cash Flow line items
    if first_col.str.contains('Operating Activity', case=False, na=False).any() or \
       first_col.str.contains('Net Cash Flow', case=False, na=False).any():
        cash_flow = table
        print(f"Cash Flow statement found at table index {i}!")
        break

# Export the historical data to CSV (TRANSPOSED)
if cash_flow is not None:
    # 1. Set the line items as the index
    cash_flow.set_index(cash_flow.columns[0], inplace=True)
    
    # 2. Transpose so Dates = rows, Line Items = columns
    cash_flow_transposed = cash_flow.T
    
    # 3. Export to CSV
    cash_flow_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_cash_flow.csv')
    print("Success! Transposed 10-year Cash Flow saved to data/raw/screener_cash_flow.csv.")
else:
    print("Error: Could not locate the Cash Flow table.")

Cash Flow statement found at table index 7!
Success! Transposed 10-year Cash Flow saved to data/raw/screener_cash_flow.csv.


In [5]:
import pandas as pd
import os

# Ensure directory exists
#os.makedirs('data/raw', exist_ok=True)

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Annual Income Statement (Profit & Loss)
income_stmt = None
for i, table in enumerate(tables):
    first_col = table.iloc[:, 0].astype(str)
    
    # 1. Look for 'Net Profit' or 'Revenue'
    has_net_profit = first_col.str.contains('Net Profit', case=False, na=False).any()
    has_revenue = first_col.str.contains('Revenue', case=False, na=False).any()
    
    # 2. Look for 'TTM' in the column headers to ensure it's the Annual table, not Quarterly
    has_ttm_column = any('TTM' in str(col).upper() for col in table.columns)
    
    if (has_net_profit or has_revenue) and has_ttm_column:
        income_stmt = table
        print(f"Annual Income Statement found at table index {i}!")
        break

# Export the historical data to CSV (TRANSPOSED)
if income_stmt is not None:
    # 1. Set the line items as the index
    income_stmt.set_index(income_stmt.columns[0], inplace=True)
    
    # 2. Transpose so Dates = rows, Line Items = columns
    income_stmt_transposed = income_stmt.T
    
    # 3. Export to CSV
    income_stmt_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_income_stmt.csv')
    print("Success! Transposed 10-year Income Statement saved to data/raw/screener_income_stmt.csv.")
else:
    print("Error: Could not locate the Annual Income Statement table.")

Error: Could not locate the Annual Income Statement table.


In [6]:
import pandas as pd
import os

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Annual Income Statement (Profit & Loss)
income_stmt = None
for i, table in enumerate(tables):
    first_col = table.iloc[:, 0].astype(str)
    
    #looking for 'net profit' or 'interest earned' (bank specific)
    is_pl_table = first_col.str.contains('Net Profit', case=False, na=False).any() or \
                  first_col.str.contains('Interest Earned', case=False, na=False).any()
    
    #check column count to differentiate annual (10+ cols) from quarterly (~6 cols)
    is_annual_table = len(table.columns) > 8
    
    if is_pl_table and is_annual_table:
        income_stmt = table
        print(f"Annual Income Statement found at table index {i}!")
        break

# Export the historical data to CSV (TRANSPOSED)
if income_stmt is not None:
    # 1. Set the line items as the index
    income_stmt.set_index(income_stmt.columns[0], inplace=True)
    
    # 2. Transpose so Dates = rows, Line Items = columns
    income_stmt_transposed = income_stmt.T
    
    # 3. Export to CSV
    income_stmt_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_income_stmt.csv')
    print("Success! Transposed 10-year Income Statement saved to data/raw/screener_income_stmt.csv.")
else:
    print("Error: Could not locate the Annual Income Statement table.")
    print("Let's look at what tables we DO have:")
    for i, t in enumerate(tables):
        print(f"\n--- Table {i} --- (Columns: {len(t.columns)})")
        print(t.iloc[:3, 0].tolist())

Annual Income Statement found at table index 0!
Success! Transposed 10-year Income Statement saved to data/raw/screener_income_stmt.csv.


In [7]:
import pandas as pd
import os

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Annual Income Statement (Profit & Loss)
income_stmt = None
for i, table in enumerate(tables):
    first_col = table.iloc[:, 0].astype(str)
    
    # Check if it's a P&L table AND if it has 'Dividend Payout' (only present in Annual view)
    is_pl_table = first_col.str.contains('Net Profit', case=False, na=False).any() or \
                  first_col.str.contains('Interest Earned', case=False, na=False).any()
                  
    is_annual_table = first_col.str.contains('Dividend Payout', case=False, na=False).any()
    
    if is_pl_table and is_annual_table:
        income_stmt = table
        print(f"Annual Income Statement found at table index {i}!")
        break

# Export the historical data to CSV (TRANSPOSED)
if income_stmt is not None:
    # 1. Set the line items as the index
    income_stmt.set_index(income_stmt.columns[0], inplace=True)
    
    # 2. Transpose so Dates = rows, Line Items = columns
    income_stmt_transposed = income_stmt.T
    
    # 3. Export to CSV
    income_stmt_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_income_stmt.csv')
    print("Success! Transposed 10-year Income Statement saved to data/raw/screener_income_stmt.csv.")
else:
    print("Error: Could not locate the Annual Income Statement table.")
    print("Let's look at what tables we DO have:")
    for i, t in enumerate(tables):
        print(f"\n--- Table {i} --- (Columns: {len(t.columns)})")
        print(t.iloc[:3, 0].tolist())

Annual Income Statement found at table index 1!
Success! Transposed 10-year Income Statement saved to data/raw/screener_income_stmt.csv.


## Data Cleaning

In [8]:
import pandas as pd
df_bal_sheet = pd.read_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_balance_sheet.csv', index_col=0)
df_cf = pd.read_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_cash_flow.csv', index_col=0)
df_income_stmt = pd.read_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_income_stmt.csv', index_col=0)

In [9]:
df_bal_sheet.head(13)

#so we have 12 years of data
#all figures are in crores (10 million) as per screener.in

,Equity Capital,Reserves,Deposits,Borrowing,Other Liabilities +,Total Liabilities,Fixed Assets +,CWIP,Investments,Other Assets +,Total Assets
Mar 2015,501,62653,450284,59478,34181,607097,3225,0,149454,454417,607097
Mar 2016,506,73798,545873,103714,38321,762212,3667,0,193634,564912,762212
Mar 2017,513,91281,643134,98416,59000,892344,4000,0,210777,677567,892344
Mar 2018,519,109080,788375,156442,48770,1103186,4008,0,238461,860717,1103186
Mar 2019,545,153128,922503,157733,58898,1292806,4369,0,289446,998991,1292806
Mar 2020,548,175810,1146207,186834,71430,1580830,4776,0,389305,1186750,1580830
Mar 2021,551,209259,1333721,177697,78279,1799507,5248,0,438823,1355435,1799507
Mar 2022,555,246772,1558003,226966,90639,2122934,6432,0,449264,1667238,2122934
Mar 2023,558,288880,1882663,256549,101783,2530432,8431,0,511582,2010419,2530432
Mar 2024,760,455636,2376887,730615,466296,4030194,12604,0,1005682,3011909,4030194


In [10]:
df_income_stmt.head(13)

,Revenue +,Interest,Expenses +,Financing Profit,Financing Margin %,Other Income +,Depreciation,Profit before tax,Tax %,Net Profit +,EPS in Rs,Dividend Payout %
Mar 2015,50666,27288,16164,7214,14%,9546,680,16079,33%,10703,10.66,19%
Mar 2016,63162,34070,20055,9037,14%,11212,738,19511,34%,12821,12.66,19%
Mar 2017,73271,38042,23856,11374,16%,12905,886,23393,35%,15317,14.91,18%
Mar 2018,85288,42381,29532,13374,16%,16057,967,28464,35%,18561,17.83,18%
Mar 2019,105161,53713,34856,16592,16%,18947,1221,34318,35%,22446,20.50,18%
Mar 2020,122189,62137,45459,14593,12%,24879,1277,38195,29%,27296,24.85,5%
Mar 2021,128552,59248,52457,16848,13%,27333,1385,42796,26%,31857,28.87,11%
Mar 2022,135936,58584,56557,20795,15%,31759,1681,50873,25%,38151,34.31,23%
Mar 2023,170754,77780,63042,29932,18%,33912,2345,61498,25%,46149,41.22,23%
Mar 2024,283649,154139,174196,-44685,-16%,124346,3092,76569,15%,65446,42.16,23%


In [11]:
df_cf.head(13)

,Cash from Operating Activity +,Cash from Investing Activity +,Cash from Financing Activity +,Net Cash Flow,Free Cash Flow,CFO/OP
Mar 2015,-21281,-800,18694,-3387,-22020,-45%
Mar 2016,-34435,-837,37815,2542,-35301,-63%
Mar 2017,17282,-1146,-5893,10242,16134,51%
Mar 2018,17214,-842,57378,73750,16377,49%
Mar 2019,-62872,-1503,23131,-41244,-64470,-72%
Mar 2020,-16869,-1403,24394,6122,-18486,-8%
Mar 2021,42476,-1823,-7321,33332,40796,73%
Mar 2022,-11960,-2051,48124,34113,-14176,4%
Mar 2023,20814,-2992,23941,41762,17390,35%
Mar 2024,19069,16600,-3983,31687,14882,38%


In [12]:
df_bal_sheet.isnull().sum()

Equity Capital         0
Reserves               0
Deposits               0
Borrowing              0
Other Liabilities +    0
Total Liabilities      0
Fixed Assets +         0
CWIP                   0
Investments            0
Other Assets +         0
Total Assets           0
dtype: int64

In [13]:
df_cf.isnull().sum()

Cash from Operating Activity +    0
Cash from Investing Activity +    0
Cash from Financing Activity +    0
Net Cash Flow                     0
Free Cash Flow                    0
CFO/OP                            0
dtype: int64

In [14]:
df_income_stmt.isnull().sum()

Revenue +             0
Interest              0
Expenses +            0
Financing Profit      0
Financing Margin %    0
Other Income +        0
Depreciation          0
Profit before tax     0
Tax %                 0
Net Profit +          0
EPS in Rs             0
Dividend Payout %     0
dtype: int64

In [15]:
#droping null columns NPA is not required for finding financial KPIs, so we can drop it
#standardize column names to remove spaces and those + signs that can cause issues in code. I've also replace spaces with underscores for consistency and easier access in code
#Cleaning function -
def clean_data(df):
    """
    Cleans raw Screener.in financial data for time-series forecasting.
    """
    # Standardize column names (remove +, lowercase, replace spaces)
    df.columns = (
        df.columns
        .str.replace('+', '', regex=False)
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
    )
    
    # Drop columns that are completely empty (like 'raw_pdf')
    df = df.dropna(axis=1, how='all')
    
    # Drop the 'TTM' row if it exists (for strict annual time-series)
    if 'ttm' in df.index:
        df = df.drop('ttm')
        
    # Convert the string index ('Mar 2024') into actual datetime objects
    # Screener uses 'Mar 2024' format, so we use '%b %Y'
    df.index = pd.to_datetime(df.index, format='%b %Y')
    
    # 5. Sort chronologically (oldest to newest)
    df = df.sort_index()

  
    
    return df

In [16]:
df_bal_sheet_clean = clean_data(df_bal_sheet)
df_cf_clean = clean_data(df_cf)
df_income_stmt_clean = clean_data(df_income_stmt)

In [17]:
# Dropping the columns with all '0' enteries 
df_bal_sheet_clean = df_bal_sheet_clean.drop(columns='cwip')
df_income_stmt_clean = df_income_stmt_clean.drop(columns='depreciation')

In [18]:
df_bal_sheet_clean.head(13)

,equity_capital,reserves,deposits,borrowing,other_liabilities,total_liabilities,fixed_assets,investments,other_assets,total_assets
2015-03-01,501,62653,450284,59478,34181,607097,3225,149454,454417,607097
2016-03-01,506,73798,545873,103714,38321,762212,3667,193634,564912,762212
2017-03-01,513,91281,643134,98416,59000,892344,4000,210777,677567,892344
2018-03-01,519,109080,788375,156442,48770,1103186,4008,238461,860717,1103186
2019-03-01,545,153128,922503,157733,58898,1292806,4369,289446,998991,1292806
2020-03-01,548,175810,1146207,186834,71430,1580830,4776,389305,1186750,1580830
2021-03-01,551,209259,1333721,177697,78279,1799507,5248,438823,1355435,1799507
2022-03-01,555,246772,1558003,226966,90639,2122934,6432,449264,1667238,2122934
2023-03-01,558,288880,1882663,256549,101783,2530432,8431,511582,2010419,2530432
2024-03-01,760,455636,2376887,730615,466296,4030194,12604,1005682,3011909,4030194


In [19]:
df_income_stmt_clean.head(13)

,revenue,interest,expenses,financing_profit,financing_margin_%,other_income,profit_before_tax,tax_%,net_profit,eps_in_rs,dividend_payout_%
2015-03-01,50666,27288,16164,7214,14%,9546,16079,33%,10703,10.66,19%
2016-03-01,63162,34070,20055,9037,14%,11212,19511,34%,12821,12.66,19%
2017-03-01,73271,38042,23856,11374,16%,12905,23393,35%,15317,14.91,18%
2018-03-01,85288,42381,29532,13374,16%,16057,28464,35%,18561,17.83,18%
2019-03-01,105161,53713,34856,16592,16%,18947,34318,35%,22446,20.50,18%
2020-03-01,122189,62137,45459,14593,12%,24879,38195,29%,27296,24.85,5%
2021-03-01,128552,59248,52457,16848,13%,27333,42796,26%,31857,28.87,11%
2022-03-01,135936,58584,56557,20795,15%,31759,50873,25%,38151,34.31,23%
2023-03-01,170754,77780,63042,29932,18%,33912,61498,25%,46149,41.22,23%
2024-03-01,283649,154139,174196,-44685,-16%,124346,76569,15%,65446,42.16,23%


In [20]:
df_cf_clean.head(13)

,cash_from_operating_activity,cash_from_investing_activity,cash_from_financing_activity,net_cash_flow,free_cash_flow,cfo/op
2015-03-01,-21281,-800,18694,-3387,-22020,-45%
2016-03-01,-34435,-837,37815,2542,-35301,-63%
2017-03-01,17282,-1146,-5893,10242,16134,51%
2018-03-01,17214,-842,57378,73750,16377,49%
2019-03-01,-62872,-1503,23131,-41244,-64470,-72%
2020-03-01,-16869,-1403,24394,6122,-18486,-8%
2021-03-01,42476,-1823,-7321,33332,40796,73%
2022-03-01,-11960,-2051,48124,34113,-14176,4%
2023-03-01,20814,-2992,23941,41762,17390,35%
2024-03-01,19069,16600,-3983,31687,14882,38%


In [21]:
import pandas as pd

def strip_percentages_v3(df):
    """
    Unconditionally strips percentages and commas from all columns
    and forces them into numeric format, bypassing any Pandas dtype traps.
    """
    df_clean = df.copy()
    
    for col in df_clean.columns:
        # Covert everything to string, strip %, commas, and spaces
        cleaned_series = (df_clean[col]
                          .astype(str)
                          .str.replace('%', '', regex=False)
                          .str.replace(',', '', regex=False)
                          .str.strip())
        
        # Force conversion back to numbers
        df_clean[col] = pd.to_numeric(cleaned_series, errors='coerce')
        
    return df_clean

# Run it on your fresh dataframes
df_cf_clean = strip_percentages_v3(df_cf_clean)
df_income_stmt_clean = strip_percentages_v3(df_income_stmt_clean)
df_bal_sheet_clean = strip_percentages_v3(df_bal_sheet_clean)



In [22]:
df_cf_clean.head()

,cash_from_operating_activity,cash_from_investing_activity,cash_from_financing_activity,net_cash_flow,free_cash_flow,cfo/op
2015-03-01,-21281,-800,18694,-3387,-22020,-45
2016-03-01,-34435,-837,37815,2542,-35301,-63
2017-03-01,17282,-1146,-5893,10242,16134,51
2018-03-01,17214,-842,57378,73750,16377,49
2019-03-01,-62872,-1503,23131,-41244,-64470,-72


In [23]:
df_bal_sheet_clean.head()

,equity_capital,reserves,deposits,borrowing,other_liabilities,total_liabilities,fixed_assets,investments,other_assets,total_assets
2015-03-01,501,62653,450284,59478,34181,607097,3225,149454,454417,607097
2016-03-01,506,73798,545873,103714,38321,762212,3667,193634,564912,762212
2017-03-01,513,91281,643134,98416,59000,892344,4000,210777,677567,892344
2018-03-01,519,109080,788375,156442,48770,1103186,4008,238461,860717,1103186
2019-03-01,545,153128,922503,157733,58898,1292806,4369,289446,998991,1292806


In [24]:
df_income_stmt_clean.head(10)

,revenue,interest,expenses,financing_profit,financing_margin_%,other_income,profit_before_tax,tax_%,net_profit,eps_in_rs,dividend_payout_%
2015-03-01,50666,27288,16164,7214,14,9546,16079,33,10703,10.66,19
2016-03-01,63162,34070,20055,9037,14,11212,19511,34,12821,12.66,19
2017-03-01,73271,38042,23856,11374,16,12905,23393,35,15317,14.91,18
2018-03-01,85288,42381,29532,13374,16,16057,28464,35,18561,17.83,18
2019-03-01,105161,53713,34856,16592,16,18947,34318,35,22446,20.50,18
2020-03-01,122189,62137,45459,14593,12,24879,38195,29,27296,24.85,5
2021-03-01,128552,59248,52457,16848,13,27333,42796,26,31857,28.87,11
2022-03-01,135936,58584,56557,20795,15,31759,50873,25,38151,34.31,23
2023-03-01,170754,77780,63042,29932,18,33912,61498,25,46149,41.22,23
2024-03-01,283649,154139,174196,-44685,-16,124346,76569,15,65446,42.16,23


### Cleaned all the data 


In [25]:
# Exporting the cleaned data to processed folder
df_income_stmt_clean.to_csv('D:\\Financial KPI Dashboard\\data\\processed\\income_clean.csv')
df_bal_sheet_clean.to_csv('D:\\Financial KPI Dashboard\\data\\processed\\balance_clean.csv')
df_cf_clean.to_csv('D:\\Financial KPI Dashboard\\data\\processed\\cashflow_clean.csv')